# Build Race Dimension
1. Read silver `races` table
2. Read silver `circuits` table
3. Join the data from `races` with `circuits` using `circuit_id`
4. Select the required columns
    - races.season
    - races.round
    - races.race_name
    - races.race_date
    - circuits.circuit_name
    - circuits.locality
    - circuits.country
5. Write the transformed data to gold `dim_races` table

In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f'{catalog_name}.{gold_schema}.dim_races'

In [0]:
circuits_silver = f'{catalog_name}.{silver_schema}.circuits'
races_silver = f'{catalog_name}.{silver_schema}.races'

#### Step 1 - Read sources table

- circuits
- reaces

In [0]:
circuits_df = spark.table(circuits_silver)
races_df = spark.table(races_silver)

#### Step 2 - Join `race` with `circuits` using `circuit_id`
Select the following column
- races.season
- races.round
- races.race_name
- races.race_date
- circuits.circuit_name
- circuits.locality
- circuits.country

In [0]:
dim_races_df = (
    races_df
        .join(
            circuits_df,
            races_df.circuit_id == circuits_df.circuit_id,
            "inner"
        )
        .select(
            races_df.season,
            races_df.round,
            races_df.race_name,
            races_df.race_date,
            circuits_df.circuit_name,
            circuits_df.locality,
            circuits_df.country_name
            )
)

#### Step 3 - Write the transformed data to the `gold` `dim races` table

In [0]:
(
    dim_races_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))